In [1]:
from glob import glob
import numpy as np
import os
from smplx import SMPLX
import torch
import matplotlib.pyplot as plt
import random

In [7]:
import os
import glob
import numpy as np
import torch
from scipy.spatial.transform import Rotation as R
from sklearn.ensemble import IsolationForest
import numpy as np

poses_files = glob.glob("videosPPmergedNoNPZ/*/*.npz")
random.shuffle(poses_files)
print(poses_files[:10])
total_frames = 0
total_frames_with_face = 0
bad_files = []
person_count_per_file = {}
output_data_per_file = {}
cpt=0
for path in poses_files:
    cpt+=1
    try:
        data = np.load(path, allow_pickle=True)
        person_keys = [key for key in data.files if key.startswith("body_")]
        num_persons = 0
        frames_in_file = 0
        people_data = {}
        frames_in_file_face=0
        for key in person_keys:                
            n_face=0
            body = data[key].item()
            if "poses" in body and "trans" in body and hasattr(body["poses"], "shape"):
                poses = body[
                    "poses"
                ]  # (N, 330) — 6 pour global_orient, le reste pour body_pose
                transl = body["trans"]  # (N, 3)

                global_orient = poses[:, :3]  # (N, 6)
                body_pose = poses[:, 3:]  # (N, 324)
                people_data[key] = {
                    "poses": body_pose,  # (N, 324)
                    "global_orient": global_orient,
                    "trans": transl,
                    "fps": body.get("fps", 30),  # Par défaut, 30 FPS
                    "expressions": body.get("expressions", None),  # Expression faciale
                }
                n = poses.shape[0]
                for frame in people_data[key]["expressions"]:
                    if frame.mean() != 0:
                        n_face +=1 

                frames_in_file += n
                frames_in_file_face += n_face
                total_frames += n
                total_frames_with_face += n_face
                num_persons += 1

        person_count_per_file[os.path.basename(path)] = {
            "num_persons": num_persons,
            "total_frames": frames_in_file,
            "total_frames_with_face": frames_in_file_face,
        }

        output_data_per_file[os.path.basename(path)] = people_data

    except Exception as e:
        bad_files.append((os.path.basename(path), str(e)))
# Résumé
print(f"\nFichiers traités : {len(person_count_per_file)}")
print(f"Total 3D pose frames: {total_frames:,}")
print(f"Total 3D face frames: {total_frames_with_face:,}")


print("\nNombre de personnes + frames (exemples) :")
for fname, stats in list(person_count_per_file.items())[:5]:
    print(f"{fname}: {stats['num_persons']} personnes, {stats['total_frames']} frames, {stats['total_frames_with_face']} frames avec visage")

if bad_files:
    print(f"\nFichiers invalides : {len(bad_files)}")
    for name, err in bad_files[:5]:
        print(f" - {name}: {err}")

['videosPPmergedNoNPZ/video_18849/video_18849_merged_scene_15.npz', 'videosPPmergedNoNPZ/video_18833/video_18833_merged_scene_1.npz', 'videosPPmergedNoNPZ/video_17675/video_17675_merged_scene_4.npz', 'videosPPmergedNoNPZ/video_15447/video_15447_merged_scene_17.npz', 'videosPPmergedNoNPZ/video_13424/video_13424_merged_scene_29.npz', 'videosPPmergedNoNPZ/video_14586/video_14586_merged_scene_8.npz', 'videosPPmergedNoNPZ/video_10262/video_10262_merged_scene_2.npz', 'videosPPmergedNoNPZ/video_18565/video_18565_merged_scene_9.npz', 'videosPPmergedNoNPZ/video_12018/video_12018_merged_scene_47.npz', 'videosPPmergedNoNPZ/video_11086/video_11086_merged_scene_16.npz']

Fichiers traités : 59351
Total 3D pose frames: 21,189,677
Total 3D face frames: 10,777,962

Nombre de personnes + frames (exemples) :
video_18849_merged_scene_15.npz: 1 personnes, 254 frames, 48 frames avec visage
video_18833_merged_scene_1.npz: 1 personnes, 282 frames, 282 frames avec visage
video_17675_merged_scene_4.npz: 6 perso